In [2]:
# Actualizar repositorios e instalar Java
!apt-get update -qq
!apt-get install openjdk-8-jdk-headless -qq -y

# Descargar Spark (usando versión estable del archivo)
!wget -q https://archive.apache.org/dist/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz
!tar xf spark-3.5.7-bin-hadoop3.tgz

# Instalar findspark
!pip install -q findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libxtst6:amd64.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package openjdk-8-jre-headless:amd64.
Preparing to unpack .../openjdk-8-jre-headless_8u472-ga-1~22.04_amd64.deb ...
Unpacking openjdk-8-jre-headless:amd64 (8u472-ga-1~22.04) ...
Selecting previously unselected package openjdk-8-jdk-headless:amd64.
Preparing to unpack .../openjdk-8-jdk-headless_8u472-ga-1~22.04_amd64.deb ...
Unpacking openjdk-8-jdk-headless:amd64 (8u472-ga-1~22.04) ...
Setting up libxtst6:amd64 (2:1.2.3-1build4) ...
Setting up openjdk-8-jre-headless:amd64 (8u472-ga-1~22.04) ...
update-alternatives: using /usr/lib/jvm

In [3]:
!pip install opendatasets
import opendatasets as od

In [4]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [10]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.7-bin-hadoop3"

In [11]:
!ls

sample_data  spark-3.5.7-bin-hadoop3  spark-3.5.7-bin-hadoop3.tgz


In [12]:
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) 
spark

In [ ]:
# from google.colab import drive  # type: ignore
# drive.mount('/content/drive')

In [13]:
# dataset_link="https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store"
dataset_link="https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store?select=2019-Nov.csv"
od.download(dataset_link)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:Your Kaggle Key:Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store


100%|██████████| 4.29G/4.29G [00:59<00:00, 77.1MB/s]

In [14]:
import os
os.chdir("ecommerce-behavior-data-from-multi-category-store")
os.listdir()

['2019-Nov.csv', '2019-Oct.csv']

# 1. Análisis profundo de los datos


## 1.1 Estructura de los datos

In [16]:
# Leer el archivo CSV con PySpark (el separador por defecto es coma)
# El archivo está en la subcarpeta que descargaste con opendatasets
df = spark.read.csv('ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv', header=True, inferSchema=True)
df.show(5)
print(f"Número de registros: {df.count()}")
print(f"Número de columnas: {len(df.columns)}")
print(f"Columnas: {df.columns}")

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|2053013555631882655|electr

In [ ]:
# Esquema del DataFrame
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [13]:
df.describe()

summary,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
count,42448764,42448764,42448764,28933155,36335756,42448764,42448764,42448762
mean,NULL,1.0549932375842676E7,2.057404237936260...,NULL,NaN,290.3236606848809,5.335371475081686E8,NULL
stddev,NULL,1.1881906970608136E7,1.843926466140411...,NULL,NaN,358.2691553394021,1.852373817465431E7,NULL
min,cart,1000978,2053013552226107603,accessories.bag,a-case,0.0,33869381,00000042-3e3f-42f...
max,view,60500010,2175419595093967522,stationery.cartrige,zyxel,2574.07,566280860,fffffc65-7ce9-435...


__Valores únicos por columna__

In [15]:
from pyspark.sql.functions import countDistinct

df.agg(
    countDistinct("event_type").alias("valores_unicos_event_type"),
    countDistinct("product_id").alias("valores_unicos_product_id"),
    countDistinct("category_id").alias("valores_unicos_category_id"),
    countDistinct("category_code").alias("valores_unicos_category_code"),
    countDistinct("brand").alias("valores_unicos_brand"),
    countDistinct("price").alias("valores_unicos_price"),
    countDistinct("user_id").alias("valores_unicos_user_id"),
).show()

+-------------------------+-------------------------+--------------------------+----------------------------+--------------------+--------------------+----------------------+
|valores_unicos_event_type|valores_unicos_product_id|valores_unicos_category_id|valores_unicos_category_code|valores_unicos_brand|valores_unicos_price|valores_unicos_user_id|
+-------------------------+-------------------------+--------------------------+----------------------------+--------------------+--------------------+----------------------+
|                        3|                   166794|                       624|                         126|                3445|               65298|               3022290|
+-------------------------+-------------------------+--------------------------+----------------------------+--------------------+--------------------+----------------------+



__Registros duplicados__

In [ ]:
total_registros = df.count()
registros_unicos = df.distinct().count()
duplicados = total_registros - registros_unicos
duplicados

In [18]:
# 1. Eliminación de registros duplicados
df_clean = df.dropDuplicates()
# print(f"Registros después de eliminar duplicados: {df_clean.count()}")


**Reemplazo de valores nulos en columnas category_code y brand** (Se les reasigna un nuevo valor a los registros con valores nulos)

In [19]:
from pyspark.sql.functions import when, col

df_clean = df_clean.withColumn(
    "category_code_clean",
    when(col("category_code").isNull(), "unknown").otherwise(col("category_code"))
)

In [20]:
df_clean = df_clean.withColumn(
    "brand_clean",
    when(col("brand").isNull(), "no_brand").otherwise(col("brand")) 
)

In [21]:
df_clean.show(10)

+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+--------------------+-----------+
|         event_time|event_type|product_id|        category_id|       category_code|  brand| price|  user_id|        user_session| category_code_clean|brand_clean|
+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+--------------------+-----------+
|2019-10-01 00:00:20|      view|   4803399|2053013554658804075|electronics.audio...|    jbl| 33.21|555428858|8a6afed4-77f8-40c...|electronics.audio...|        jbl|
|2019-10-01 00:03:30|      view|  27700106|2053013560086233771|construction.tool...|    leo| 64.35|515630204|f9cc0313-5572-489...|construction.tool...|        leo|
|2019-10-01 00:18:54|      view|   1004625|2053013555631882655|electronics.smart...|    fly| 43.73|513840435|f13c75a5-7f6e-446...|electronics.smart...|        fly|
|2019-10-01 01:4

In [51]:
df_clean.cache()

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category_code_clean,brand_clean
2019-10-01 00:00:20,view,4803399,2053013554658804075,electronics.audio...,jbl,33.21,555428858,8a6afed4-77f8-40c...,electronics.audio...,jbl
2019-10-01 00:03:30,view,27700106,2053013560086233771,construction.tool...,leo,64.35,515630204,f9cc0313-5572-489...,construction.tool...,leo
2019-10-01 00:18:54,view,1004625,2053013555631882655,electronics.smart...,fly,43.73,513840435,f13c75a5-7f6e-446...,electronics.smart...,fly
2019-10-01 01:47:33,view,2501143,2053013564003713919,appliances.kitche...,artel,36.01,515173040,36c447da-c564-d2e...,appliances.kitche...,artel
2019-10-01 02:17:07,view,1003050,2053013555631882655,electronics.smart...,samsung,617.52,540508933,29726f35-e3c6-49c...,electronics.smart...,samsung
2019-10-01 02:18:49,view,1003991,2053013555631882655,electronics.smart...,lg,216.2,550346297,8d8694d3-271a-4d0...,electronics.smart...,lg
2019-10-01 02:19:19,view,1005115,2053013555631882655,electronics.smart...,apple,975.57,547806687,5b7c5da1-9a61-425...,electronics.smart...,apple
2019-10-01 02:22:28,view,1005115,2053013555631882655,electronics.smart...,apple,975.57,534540964,a3e62f46-6e32-494...,electronics.smart...,apple
2019-10-01 02:22:39,view,1004659,2053013555631882655,electronics.smart...,samsung,787.18,550422448,f9e45568-7f95-436...,electronics.smart...,samsung
2019-10-01 02:25:13,view,5100562,2053013553341792533,electronics.clocks,apple,296.8,512736046,f0a6fb4a-df23-49b...,electronics.clocks,apple


## 1.2 Análisis temporal

In [35]:
from pyspark.sql.functions import min, max

df_clean.select(
    min("event_time").alias("fecha_minima"),
    max("event_time").alias("fecha_maxima")
).show()

+-------------------+-------------------+
|       fecha_minima|       fecha_maxima|
+-------------------+-------------------+
|2019-10-01 00:00:00|2019-10-31 23:59:59|
+-------------------+-------------------+



In [52]:
from pyspark.sql.functions import to_date, hour, dayofweek, date_format

# Eventos por día de la semana
df_clean.withColumn("dia_semana", dayofweek(col("event_time"))) \
    .groupBy("dia_semana") \
    .count() \
    .orderBy("dia_semana") \
    .show()

+----------+-------+
|dia_semana|  count|
+----------+-------+
|         1|5851611|
|         2|5317662|
|         3|6797348|
|         4|6648353|
|         5|6376063|
|         6|5824835|
|         7|5602672|
+----------+-------+



In [53]:
# Horas del día con más eventos
df_clean.withColumn("hora_dia", hour(col("event_time"))) \
    .groupBy("hora_dia") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(10)
    

+--------+-------+
|hora_dia|  count|
+--------+-------+
|      16|3053226|
|      15|2979082|
|      17|2732443|
|      14|2676546|
|       8|2387991|
|      13|2353321|
|       9|2349336|
|       7|2333480|
|      10|2295245|
|       6|2266954|
+--------+-------+
only showing top 10 rows



In [54]:
# Estadísticas de eventos por día
eventos_por_dia = df_clean.withColumn("fecha", to_date(col("event_time"))) \
    .groupBy("fecha") \
    .count() 
eventos_por_dia.describe("count").show()

+-------+------------------+
|summary|             count|
+-------+------------------+
|  count|                31|
|   mean|1368340.1290322582|
| stddev|120140.19894682542|
|    min|           1126624|
|    max|           1638290|
+-------+------------------+



## 1.3 Análisis de comportamiento de usuarios

**Distribución de tipos de eventos**

In [21]:
distribucion_eventos = df_clean.groupBy("event_type").count() \
    .withColumn("porcentaje", (col("count") / df_clean.count() * 100)) \
    .orderBy(col("count").desc())
distribucion_eventos.show()

+----------+--------+------------------+
|event_type|   count|        porcentaje|
+----------+--------+------------------+
|      view|40777328|  96.1308997310233|
|      cart|  898443|2.1180429955351605|
|  purchase|  742773|1.7510572734415402|
+----------+--------+------------------+



In [56]:
# Calculo de tasas de conversión globales
eventos_totales = df_clean.count()
views = df_clean.filter(col("event_type") == "view").count()
carts = df_clean.filter(col("event_type") == "cart").count()
purchases = df_clean.filter(col("event_type") == "purchase").count()

print("Tasas de conversión globales: ")
print(f"Views: {views:,} ({views / eventos_totales * 100:.2f}%)")
print(f"Carts: {carts:,} ({carts / eventos_totales * 100:.2f}%)")
print(f"Purchase: {purchases:,} ({purchases / eventos_totales * 100:.2f}%)")
print(f"\nTasa cart/view: {carts / views * 100:.2f}%")
print(f"Tasa purchase/view: {purchases/views*100:.2f}%")
print(f"Tasa purchase/cart: {purchases/carts*100:.2f}%")

Tasas de conversión globales: 
Views: 40,777,328 (96.13%)
Carts: 898,443 (2.12%)
Purchase: 742,773 (1.75%)

Tasa cart/view: 2.20%
Tasa purchase/view: 1.82%
Tasa purchase/cart: 82.67%


**Métricas de actividad por usuario**

In [24]:
# Distribución de eventos por usuario
eventos_por_usuario = df_clean.groupBy("user_id").count().withColumnRenamed("count", "total_eventos")
eventos_por_usuario.describe("total_eventos").show()

+-------+-----------------+
|summary|    total_eventos|
+-------+-----------------+
|  count|          3022290|
|   mean| 14.0352328863213|
| stddev|32.75705325743598|
|    min|                1|
|    max|             7436|
+-------+-----------------+



In [58]:
# Distribución en percentiles
eventos_por_usuario.selectExpr(
    "percentile_approx(total_eventos, 0.25) as p25",
    "percentile_approx(total_eventos, 0.50) as p50_median",
    "percentile_approx(total_eventos, 0.75) as p75",
    "percentile_approx(total_eventos, 0.90) as p90",
    "percentile_approx(total_eventos, 0.95) as p95",
    "percentile_approx(total_eventos, 0.99) as p99"
).show()

+---+----------+---+---+---+---+
|p25|p50_median|p75|p90|p95|p99|
+---+----------+---+---+---+---+
|  2|         4| 13| 34| 57|140|
+---+----------+---+---+---+---+



**Eventos por tipo**

In [29]:
from pyspark.sql.functions import sum as spark_sum

eventos_por_tipo_usuario = df_clean.groupBy("user_id", "event_type").count() \
    .groupBy("user_id").pivot("event_type").sum("count") \
    .fillna(0)
eventos_por_tipo_usuario.show(5)

+---------+----+--------+----+
|  user_id|cart|purchase|view|
+---------+----+--------+----+
|514700043|   0|       0|   4|
|523367237|   1|       1|  53|
|519705698|   0|       0|  89|
|519798460|   1|       2| 102|
|547283407|   3|       3|  13|
+---------+----+--------+----+
only showing top 5 rows



In [60]:
# Estadísticas de cada tipo de evento por usuario
eventos_por_tipo_usuario.describe("view").show()
eventos_por_tipo_usuario.describe("cart").show()
eventos_por_tipo_usuario.describe("purchase").show()

+-------+------------------+
|summary|              view|
+-------+------------------+
|  count|           3022290|
|   mean|13.492195652965135|
| stddev|31.855348368395525|
|    min|                 0|
|    max|              7436|
+-------+------------------+

+-------+------------------+
|summary|              cart|
+-------+------------------+
|  count|           3022290|
|   mean|0.2972722670557756|
| stddev|1.5011636291005397|
|    min|                 0|
|    max|               356|
+-------+------------------+

+-------+------------------+
|summary|          purchase|
+-------+------------------+
|  count|           3022290|
|   mean|0.2457649663003881|
| stddev|1.4093212679874512|
|    min|                 0|
|    max|               321|
+-------+------------------+



**Compradores vs navegadores**

In [61]:
usuarios_con_compras = df_clean.filter(col("event_type") == "purchase").select("user_id").distinct()

total_usuarios = df_clean.select("user_id").distinct().count()
compradores = usuarios_con_compras.count()
navegadores = total_usuarios - compradores

print(f"Total de usuarios: {total_usuarios:,}")
print(f"Compradores: {compradores:,} ({compradores / total_usuarios * 100:.2f}%)")
print(f"Solo navegadores: {navegadores:,} ({navegadores / total_usuarios * 100:.2f}%)")
print(f"Ratio compradores/navegadores: {compradores / navegadores:.2f}")

Total de usuarios: 3,022,290
Compradores: 347,118 (11.49%)
Solo navegadores: 2,675,172 (88.51%)
Ratio compradores/navegadores: 0.13


**Análisis de compras por usuario**

In [62]:
from pyspark.sql.functions import count, avg

compras_por_usuario = df_clean.filter(col("event_type") == "purchase") \
    .groupBy("user_id").agg(
        count("*").alias("num_compras"),
        spark_sum("price").alias("total_gastado"),
        avg("price").alias("gasto_promedio"),
        min("price").alias("gasto_minimo"),
        max("price").alias("gasto_maximo")
    )
    
print("Estadísticas de compras por usuario:")
compras_por_usuario.describe().show()

Estadísticas de compras por usuario:
+-------+--------------------+------------------+------------------+-----------------+-----------------+------------------+
|summary|             user_id|       num_compras|     total_gastado|   gasto_promedio|     gasto_minimo|      gasto_maximo|
+-------+--------------------+------------------+------------------+-----------------+-----------------+------------------+
|  count|              347118|            347118|            347118|           347118|           347118|            347118|
|   mean| 5.359970205283938E8|2.1398285309318443| 662.4064803035074|278.0298408672455|241.1867638670545|320.26698992851243|
| stddev|1.8498600835749354E7|3.6387369214359544|2074.2141910618348|311.2153181859061|297.9926696586466| 360.8857670524742|
|    min|           264649825|                 1|              0.88|             0.88|             0.77|              0.88|
|    max|           566278294|               321|265569.51999999984|          2574.04|         

## Tasa de conversión por usuario

In [30]:
# Calculo de tasa de conversión por usuario
conversion_por_usuario = eventos_por_tipo_usuario.withColumn(
    "tasa_de_conversion",
    when(col("view") > 0, col("purchase") / col("view") * 100).otherwise(0)
).withColumn(
    "cart_rate",
    when(col("view") > 0, col("cart") / col("view") * 100).otherwise(0)
)

print("Distribución de tasas de conversión por usuario:")
conversion_por_usuario.select("tasa_de_conversion", "cart_rate").describe().show()

Distribución de tasas de conversión por usuario:
+-------+------------------+------------------+
|summary|tasa_de_conversion|         cart_rate|
+-------+------------------+------------------+
|  count|           3022290|           3022290|
|   mean| 2.003747909988533| 2.527292187227874|
| stddev| 8.888059905115897|13.473155834735465|
|    min|               0.0|               0.0|
|    max|             200.0|            2800.0|
+-------+------------------+------------------+



## 1.4 Análisis de productos y categorías

**Catálogo de productos**

In [34]:
# Total de productos únicos
total_productos = df_clean.select("product_id").distinct().count()
print(f"Total de productos únicos: {total_productos:,}")

Total de productos únicos: 166,794


**Tasa de conversión por producto**

In [35]:
views_por_producto = df_clean.filter(col("event_type") == "view") \
    .groupBy("product_id").count().withColumnRenamed("count", "num_views") 
    
compras_por_producto = df_clean.filter(col("event_type") == "purchase") \
    .groupBy("product_id").count().withColumnRenamed("count", "num_purchases")
    
conversion_producto = views_por_producto.join(compras_por_producto, "product_id", "left") \
    .fillna(0, subset=["num_purchases"]) \
    .withColumn(
        "tasa_conversion_producto",
        when(col("num_views") > 0, col("num_purchases") / col("num_views") * 100).otherwise(0)
    )
    
print("Distribución de tasas de conversión por producto:")
conversion_producto.describe("tasa_conversion_producto").show()

Distribución de tasas de conversión por producto:
+-------+------------------------+
|summary|tasa_conversion_producto|
+-------+------------------------+
|  count|                  166794|
|   mean|      0.5676354841306908|
| stddev|       2.236626626845516|
|    min|                     0.0|
|    max|                   100.0|
+-------+------------------------+



**Categorías más populares**

In [36]:
print("Top 10 categorías más vistas:")
df_clean.filter(col("event_type") == "view").groupBy("category_code_clean").count() \
    .orderBy(col("count").desc()).show(10, truncate=False)
    
print("Top 10 categorías con más compras:")
df_clean.filter(col("event_type") == "purchase").groupBy("category_code_clean").count() \
    .orderBy(col("count").desc()).show(10, truncate=False)

print("Top categorías por ganancias:")
df_clean.filter(col("event_type") == "purchase") \
    .groupBy("category_code_clean") \
    .agg(spark_sum("price").alias("total_ganancias")) \
    .orderBy(col("total_ganancias").desc()) \
    .show(10, truncate=False)

Top 10 categorías más vistas:
+--------------------------------+--------+
|category_code_clean             |count   |
+--------------------------------+--------+
|unknown                         |13235871|
|electronics.smartphone          |10618648|
|electronics.clocks              |1272733 |
|computers.notebook              |1106321 |
|electronics.video.tv            |1055922 |
|electronics.audio.headphone     |1018478 |
|appliances.kitchen.refrigerators|863358  |
|appliances.kitchen.washer       |831234  |
|appliances.environment.vacuum   |772001  |
|apparel.shoes                   |759637  |
+--------------------------------+--------+
only showing top 10 rows

Top 10 categorías con más compras:
+--------------------------------+------+
|category_code_clean             |count |
+--------------------------------+------+
|electronics.smartphone          |337979|
|unknown                         |173411|
|electronics.audio.headphone     |30501 |
|electronics.video.tv            |21561 |

**Lealtad a la marca**

In [ ]:
from pyspark.sql.functions import countDistinct

# Usuarios que compran de una sola marca vs varias
marcas_por_usuario = df_clean.filter(col("event_type") == "purchase").groupBy("user_id") \
    .agg(countDistinct("brand_clean").alias("num_marcas_compradas"))

print("Lealtad a la marca")
marcas_por_usuario.describe("num_marcas_compradas").show()

# usuarios leales (1 sola marca) y diversos (varias marcas)
usuarios_leales = marcas_por_usuario.filter(col("num_marcas_compradas") == 1).count()
usuarios_diversos = marcas_por_usuario.filter(col("num_marcas_compradas") > 1).count()

print(f"Compradores de una sola marca: {usuarios_leales:,} ({usuarios_leales / (usuarios_leales + usuarios_diversos) * 100:.2f}%)")
print(f"Compradores de varias marcas: {usuarios_diversos:,} ({usuarios_diversos / (usuarios_leales + usuarios_diversos) * 100:.2f}%)")

Lealtad a la marca
+-------+--------------------+
|summary|num_marcas_compradas|
+-------+--------------------+
|  count|              347118|
|   mean|  1.3341054050783883|
| stddev|  0.8559955198932354|
|    min|                   1|
|    max|                  30|
+-------+--------------------+

Compradores de una sola marca: 273,457 (78.78%)
Compradores de varias marcas: 73,661 (21.22%)
